# Ejercicios — Clase 7: LOB Modeling Examples

| Tier | Ejercicios | Cuándo |
|------|-----------|--------|
| **Núcleo** | 1–5 | Obligatorio en clase |
| **Si vamos bien** | 6–7 | Si el ritmo lo permite |
| **Bonus / casa** | 8–10 | Tarea o alumnos adelantados |

Los datos están en `../data/lob_modeling_features.csv` (generado por `lesson.ipynb`).

**Estos ejercicios producen gráficos.** El resultado correcto no es solo un número — es también la visualización.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import warnings; warnings.filterwarnings('ignore')

plt.style.use('dark_background')
CYAN, GREEN, RED, AMBER, MUTED = '#22d3ee', '#4ade80', '#f87171', '#f59e0b', '#a1a1aa'

FEATS = ['imbalance', 'spread_pct', 'wmid', 'depth_ratio',
         'imbalance_mean_5', 'mid_momentum_5', 'imbalance_std_5']

---
## Ejercicio 0 — Antes de modelar: ¿limit o market?

*Sin código. Solo reflexión.*

**a)** Tienes un LOB de BTC con imbalance = 0.78. El precio ha subido $15 en los últimos 5 snapshots. ¿Colocarías una limit bid o una market buy? ¿Por qué?

**b)** Un compañero dice: "nuestro modelo tiene 55% de accuracy — no vale nada". ¿Estás de acuerdo? ¿Qué información adicional necesitarías para evaluar si tiene valor económico?

**c)** ¿Por qué un árbol de decisión sin limitar la profundidad tiene 100% de accuracy en training y ~47% en test? Explícalo en dos frases sin usar la palabra 'overfitting'.

---
## ── NÚCLEO ──────────────────────────────────────────────────────

## Ejercicio 1 — Rolling imbalance: visualizar la memoria del LOB

Carga `../data/lob_modeling_features.csv`. Crea un gráfico con dos líneas sobre los primeros 100 snapshots:
- `imbalance` en azul claro con baja opacidad
- `imbalance_mean_5` en ámbar con mayor grosor

Asigna `smoothing_effect` = desviación estándar de `imbalance_mean_5` dividida por la de `imbalance`.
Debería ser menor que 1 (la media rolling es más suave).

In [ ]:
df = None              # ← carga el CSV
smoothing_effect = None  # ← ratio de desviaciones estándar

In [ ]:
# Validador 1
assert df is not None, "df no está cargado"
assert len(df) == 484, f"Se esperaban 484 filas, hay {len(df)}"
assert 'imbalance_mean_5' in df.columns, "Falta 'imbalance_mean_5'"
assert smoothing_effect is not None, "smoothing_effect no definido"
assert 0 < smoothing_effect < 1, f"smoothing_effect debe ser < 1 (es {smoothing_effect:.4f})"
assert abs(smoothing_effect - df['imbalance_mean_5'].std() / df['imbalance'].std()) < 1e-6
print('✓ Ejercicio 1 correcto')
print(f'  La media rolling reduce la std en {(1-smoothing_effect)*100:.1f}%')

In [ ]:
# Solución guiada 1
df = pd.read_csv('../data/lob_modeling_features.csv')
smoothing_effect = df['imbalance_mean_5'].std() / df['imbalance'].std()

fig, ax = plt.subplots(figsize=(12, 3))
N = 100
ax.plot(df['imbalance'].values[:N], color=CYAN, alpha=0.35, linewidth=1, label='imbalance (t)')
ax.plot(df['imbalance_mean_5'].values[:N], color=AMBER, linewidth=2.5, label='imbalance_mean_5')
ax.axhline(0.5, color=MUTED, linestyle='--', linewidth=0.8)
ax.set_title(f'Rolling mean suaviza el ruido — std reducida en {(1-smoothing_effect)*100:.1f}%', color='white')
ax.legend(labelcolor='white'); ax.set_facecolor('#18181b')
fig.patch.set_facecolor('#09090b'); plt.tight_layout(); plt.show()

---
## Ejercicio 2 — Momentum: la velocidad del precio

Crea un gráfico de barras de `mid_momentum_5` para los primeros 80 snapshots:
- Barras verdes para momentum positivo (precio subiendo)
- Barras rojas para momentum negativo (precio bajando)

Calcula `pct_positive_momentum`: porcentaje de snapshots con momentum > 0.

In [ ]:
pct_positive_momentum = None  # ← float entre 0 y 1

In [ ]:
# Validador 2
assert pct_positive_momentum is not None
assert 0.3 < pct_positive_momentum < 0.7, f"Fuera de rango: {pct_positive_momentum:.4f}"
expected = float((df['mid_momentum_5'] > 0).mean())
assert abs(pct_positive_momentum - expected) < 1e-6, f"Incorrecto: {pct_positive_momentum:.4f} (esperado {expected:.4f})"
print('✓ Ejercicio 2 correcto')
print(f'  {pct_positive_momentum:.1%} de snapshots tienen momentum positivo — mercado de tendencia alcista leve')

In [ ]:
# Solución guiada 2
pct_positive_momentum = float((df['mid_momentum_5'] > 0).mean())
N = 80; mom = df['mid_momentum_5'].values[:N]
cols = [GREEN if v > 0 else RED for v in mom]

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(N), mom, color=cols, alpha=0.75, width=1)
ax.axhline(0, color=MUTED, linewidth=0.8)
ax.set_title('mid_momentum_5: verde = precio subiendo, rojo = precio bajando', color='white')
ax.set_facecolor('#18181b'); fig.patch.set_facecolor('#09090b')
plt.tight_layout(); plt.show()

---
## Ejercicio 3 — Split temporal y LR baseline

Haz el split temporal 70/30 sobre `df`. Verifica el tamaño.  
Entrena un `LogisticRegression` con los 7 features temporales sobre `direction`.  
Asigna `acc_lr_temporal` (test accuracy).

In [ ]:
train_df = None
test_df  = None
acc_lr_temporal = None

In [ ]:
# Validador 3
assert train_df is not None and test_df is not None
assert len(train_df) == 338, f"n_train esperado 338, es {len(train_df)}"
assert len(test_df)  == 146, f"n_test esperado 146, es {len(test_df)}"
assert train_df.index.max() < test_df.index.min(), "Split debe respetar el orden temporal"
assert acc_lr_temporal is not None
assert abs(acc_lr_temporal - 0.5548) < 1e-3, f"Incorrecto: {acc_lr_temporal:.4f} (esperado ~0.5548)"
print('✓ Ejercicio 3 correcto')
print(f'  LR temporal: {acc_lr_temporal:.1%}  — mejora sobre L6 baseline (49.3%)')

In [ ]:
# Solución guiada 3
split = int(len(df) * 0.7)
train_df = df.iloc[:split].copy()
test_df  = df.iloc[split:].copy()

X_tr = train_df[FEATS].values; y_tr = train_df['direction'].values
X_te = test_df[FEATS].values;  y_te = test_df['direction'].values

lr = LogisticRegression(random_state=42, max_iter=1000).fit(X_tr, y_tr)
acc_lr_temporal = lr.score(X_te, y_te)
print(f'LR temporal test: {acc_lr_temporal:.4f}')

---
## Ejercicio 4 — Demo del overfitting con DecisionTree

Entrena **dos** árboles de decisión:
1. `dt_none`: sin límite de profundidad (`max_depth=None`)
2. `dt_2`: con `max_depth=2`

Asigna las cuatro accuracies: `acc_dt_none_train`, `acc_dt_none_test`, `acc_dt2_train`, `acc_dt2_test`.

Luego crea un gráfico de barras agrupadas comparando train vs test para ambos modelos.

In [ ]:
acc_dt_none_train = None
acc_dt_none_test  = None
acc_dt2_train     = None
acc_dt2_test      = None

In [ ]:
# Validador 4
for name, val, expected in [
    ('acc_dt_none_train', acc_dt_none_train, 1.0),
    ('acc_dt_none_test',  acc_dt_none_test,  0.4726),
    ('acc_dt2_train',     acc_dt2_train,     0.5710),
    ('acc_dt2_test',      acc_dt2_test,      0.5274),
]:
    assert val is not None, f"{name} no definido"
    assert abs(val - expected) < 1e-3, f"{name} incorrecto: {val:.4f} (esperado {expected:.4f})"
assert acc_dt_none_train - acc_dt_none_test > 0.4, \
    "La brecha train-test del DT sin podar debe ser > 40pp — ¿usaste max_depth=None?"
print('✓ Ejercicio 4 correcto')
print(f'  DT sin podar: train={acc_dt_none_train:.0%}, test={acc_dt_none_test:.0%}  — brecha: {(acc_dt_none_train-acc_dt_none_test)*100:.0f}pp')
print(f'  DT depth=2:   train={acc_dt2_train:.0%},  test={acc_dt2_test:.0%}')

In [ ]:
# Solución guiada 4
dt_none = DecisionTreeClassifier(random_state=42).fit(X_tr, y_tr)
dt_2    = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_tr, y_tr)

acc_dt_none_train = dt_none.score(X_tr, y_tr)
acc_dt_none_test  = dt_none.score(X_te, y_te)
acc_dt2_train     = dt_2.score(X_tr, y_tr)
acc_dt2_test      = dt_2.score(X_te, y_te)

fig, ax = plt.subplots(figsize=(8, 4.5))
x, w = np.arange(2), 0.35
b1 = ax.bar(x-w/2, [acc_dt_none_train*100, acc_dt2_train*100], w, label='Train', color=GREEN, alpha=0.8)
b2 = ax.bar(x+w/2, [acc_dt_none_test*100,  acc_dt2_test*100],  w, label='Test',  color=CYAN,  alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(['DT sin podar\n(max_depth=None)', 'DT depth=2'], color='white')
ax.set_ylim(35, 110); ax.set_ylabel('Accuracy (%)', color=MUTED)
ax.set_title('Overfitting: DT sin podar memoriza el train, falla en test', color='white')
for b, v in zip(b1, [acc_dt_none_train, acc_dt2_train]):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f'{v:.0%}', ha='center', color=GREEN, fontweight='bold')
for b, v in zip(b2, [acc_dt_none_test, acc_dt2_test]):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f'{v:.0%}', ha='center', color=CYAN, fontweight='bold')
ax.legend(labelcolor='white'); ax.set_facecolor('#18181b'); fig.patch.set_facecolor('#09090b')
plt.tight_layout(); plt.show()

---
## Ejercicio 5 — RandomForest + feature importance

Entrena un `RandomForestClassifier(n_estimators=100, random_state=42)` sobre `direction`.  
Asigna `acc_rf_test`. Luego crea un gráfico horizontal de barras con las feature importances, ordenado de mayor a menor.

Asigna `best_feature` = nombre del feature con mayor importancia.

In [ ]:
acc_rf_test  = None
best_feature = None

In [ ]:
# Validador 5
assert acc_rf_test is not None, "acc_rf_test no definido"
assert abs(acc_rf_test - 0.5274) < 1e-3, f"Incorrecto: {acc_rf_test:.4f} (esperado ~0.5274)"
assert best_feature in FEATS, f"best_feature debe ser uno de {FEATS}"
assert best_feature == 'depth_ratio', f"Esperado 'depth_ratio', es '{best_feature}'"
print('✓ Ejercicio 5 correcto')
print(f'  RF test: {acc_rf_test:.1%}  |  feature más importante: {best_feature}')
print('  Nota: LR temporal (55.5%) supera al RF (52.7%) con tan pocos datos.')

In [ ]:
# Solución guiada 5
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
acc_rf_test = rf.score(X_te, y_te)
best_feature = FEATS[rf.feature_importances_.argmax()]

imp = sorted(zip(FEATS, rf.feature_importances_), key=lambda x: x[1])
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh([f for f,_ in imp], [v for _,v in imp], color=CYAN, alpha=0.8)
for bar, val in zip(bars, [v for _,v in imp]):
    ax.text(val+0.002, bar.get_y()+bar.get_height()/2, f'{val:.3f}', va='center', color=CYAN, fontsize=9)
ax.set_title('Feature importances — señal distribuida, ninguno domina', color='white')
ax.set_facecolor('#18181b'); fig.patch.set_facecolor('#09090b')
plt.tight_layout(); plt.show()

---
## ── SI VAMOS BIEN ────────────────────────────────────────────────

## Ejercicio 6 — Curva de complejidad

Entrena árboles con `max_depth` en `[1, 2, 3, 4, 5, 6, 8, 10, 20]`.  
Para cada uno, registra train y test accuracy.  
Crea un gráfico de líneas con ambas curvas.

Asigna `best_depth` = el `max_depth` que da mejor test accuracy.

In [ ]:
depths_to_test = [1, 2, 3, 4, 5, 6, 8, 10, 20]
best_depth = None   # ← int

In [ ]:
# Validador 6
assert best_depth is not None, "best_depth no definido"
assert best_depth in depths_to_test, f"best_depth debe estar en {depths_to_test}"
# Verificamos con nuestros datos
te_accs = [DecisionTreeClassifier(max_depth=d,random_state=42).fit(X_tr,y_tr).score(X_te,y_te) for d in depths_to_test]
expected_best = depths_to_test[np.argmax(te_accs)]
assert best_depth == expected_best, f"best_depth incorrecto: {best_depth} (esperado {expected_best})"
print('✓ Ejercicio 6 correcto')
print(f'  Mejor depth: {best_depth}  (test acc: {max(te_accs):.1%})')
print('  La curva muestra: a partir de depth=2, el test no mejora — solo el train.')

In [ ]:
# Solución guiada 6
tr_accs, te_accs = [], []
for d in depths_to_test:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr, y_tr)
    tr_accs.append(dt.score(X_tr, y_tr))
    te_accs.append(dt.score(X_te, y_te))

best_depth = depths_to_test[np.argmax(te_accs)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(depths_to_test, [v*100 for v in tr_accs], '-o', color=GREEN, lw=2, label='Train')
ax.plot(depths_to_test, [v*100 for v in te_accs], '-o', color=RED,   lw=2, label='Test')
ax.fill_between(depths_to_test, [v*100 for v in tr_accs], [v*100 for v in te_accs], alpha=0.1, color=RED)
ax.axvline(best_depth, color=CYAN, linestyle='--', lw=1.2, label=f'Mejor depth={best_depth}')
ax.set_xlabel('max_depth', color=MUTED); ax.set_ylabel('Accuracy (%)', color=MUTED)
ax.set_title('Curva de complejidad: la brecha roja es el overfitting', color='white')
ax.legend(labelcolor='white'); ax.set_facecolor('#18181b'); fig.patch.set_facecolor('#09090b')
plt.tight_layout(); plt.show()

---
## Ejercicio 7 — Fill probability: target, modelo y visualización

1. Calcula `fill_rate_test` = fracción de fill_in_3 = 1 en el test set
2. Entrena `RandomForestClassifier(n_estimators=100, random_state=42)` sobre `fill_in_3`
3. Asigna `acc_rf_fill` (test accuracy)
4. Crea un gráfico de buckets de imbalance vs fill rate empírica (5 buckets)

In [ ]:
fill_rate_test = None
acc_rf_fill    = None

In [ ]:
# Validador 7
assert fill_rate_test is not None and acc_rf_fill is not None
assert abs(fill_rate_test - test_df['fill_in_3'].mean()) < 1e-6
assert abs(acc_rf_fill - 0.5479) < 1e-3, f"acc_rf_fill incorrecto: {acc_rf_fill:.4f} (esperado ~0.5479)"
baseline = max(test_df['fill_in_3'].mean(), 1-test_df['fill_in_3'].mean())
assert acc_rf_fill > baseline - 0.01, "RF fill debe ser cercano o mejor al baseline"
print('✓ Ejercicio 7 correcto')
print(f'  Fill rate test: {fill_rate_test:.1%}  |  RF acc: {acc_rf_fill:.1%}  |  Baseline: {baseline:.1%}')
print(f'  Mejora: +{(acc_rf_fill-baseline)*100:.1f}pp sobre mayoría')

In [ ]:
# Solución guiada 7
y_tr_f = train_df['fill_in_3'].values
y_te_f = test_df['fill_in_3'].values

rf_fill = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr_f)
fill_rate_test = float(y_te_f.mean())
acc_rf_fill    = rf_fill.score(X_te, y_te_f)

# Gráfico de buckets
df['imb_bucket'] = pd.cut(df['imbalance'], bins=[0,.35,.45,.55,.65,1.],
                           labels=['< 0.35','0.35–0.45','0.45–0.55','0.55–0.65','> 0.65'])
rates = df.groupby('imb_bucket', observed=True)['fill_in_3'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
colors_ = [GREEN if v < 0.55 else RED for v in rates]
bars = ax.bar(range(len(rates)), [v*100 for v in rates], color=colors_, alpha=0.8)
ax.set_xticks(range(len(rates))); ax.set_xticklabels(rates.index, color='white')
ax.set_ylabel('Fill rate (%)', color=MUTED)
ax.set_title('Fill rate empírica por imbalance — bajo imbalance = más fills', color='white')
for bar, v in zip(bars, rates):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{v:.0%}',
            ha='center', color='white', fontsize=10, fontweight='bold')
ax.set_facecolor('#18181b'); fig.patch.set_facecolor('#09090b')
plt.tight_layout(); plt.show()

---
## ── BONUS / CASA ─────────────────────────────────────────────────

## Ejercicio 8 — Calibración: ¿las probabilidades del modelo tienen sentido?

Un modelo bien calibrado: cuando predice P(fill)=0.7, debería ocurrir fill ~70% del tiempo.

Usa `rf_fill.predict_proba(X_te)[:, 1]` para obtener las probabilidades predichas.  
Divide en 5 buckets de probabilidad ([0,0.2), [0.2,0.4), ..., [0.8,1.0]).  
Para cada bucket, calcula la fill rate real.  
Crea un gráfico con:
- Barras = fill rate real por bucket
- Línea diagonal (0→1) = calibración perfecta

In [ ]:
# ← implementa aquí

In [ ]:
# Solución guiada 8
proba_te = rf_fill.predict_proba(X_te)[:, 1]
proba_df = pd.DataFrame({'proba': proba_te, 'actual': y_te_f})
proba_df['bucket'] = pd.cut(proba_df['proba'], bins=5)
calib = proba_df.groupby('bucket', observed=True).agg(real_rate=('actual','mean'), mid_prob=('proba','mean'), count=('actual','count'))

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(range(len(calib)), calib['real_rate'], color=CYAN, alpha=0.75, label='Fill rate real')
ax.plot([0, len(calib)-1], [calib['real_rate'].iloc[0], calib['real_rate'].iloc[-1]],
        color=AMBER, lw=2, linestyle='--', label='Tendencia')
ax.set_xticks(range(len(calib)))
ax.set_xticklabels([f'{b.left:.1f}–{b.right:.1f}' for b in calib.index], fontsize=8, color='white')
ax.set_xlabel('Bucket de probabilidad predicha', color=MUTED)
ax.set_ylabel('Fill rate real', color=MUTED)
ax.set_title('Calibración: prob predicha vs fill rate real', color='white')
ax.set_ylim(0, 1); ax.legend(labelcolor='white')
ax.set_facecolor('#18181b'); fig.patch.set_facecolor('#09090b')
plt.tight_layout(); plt.show()
print(calib[['real_rate','count']].round(3))

---
## Ejercicio 9 — Comparación exhaustiva: todos los modelos, los dos targets

Crea una tabla resumen con todos los modelos entrenados en dirección y fill_in_3:
- LR temporal, DT sin podar, DT depth=2, RF
- Para cada uno: train acc y test acc

Visualiza como heatmap con `ax.imshow()`: filas = modelos, columnas = (train dir, test dir, train fill, test fill).

In [ ]:
# ← implementa aquí

In [ ]:
# Solución guiada 9
lr  = LogisticRegression(random_state=42, max_iter=1000).fit(X_tr, y_tr)
dt0 = DecisionTreeClassifier(random_state=42).fit(X_tr, y_tr)
dt2 = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_tr, y_tr)
rf  = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)

lr_f  = LogisticRegression(random_state=42, max_iter=1000).fit(X_tr, y_tr_f)
dt0_f = DecisionTreeClassifier(random_state=42).fit(X_tr, y_tr_f)
dt2_f = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_tr, y_tr_f)
rf_f2 = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr_f)

model_names = ['LR temporal', 'DT sin podar', 'DT depth=2', 'RF n=100']
data = np.array([
    [lr.score(X_tr,y_tr),  lr.score(X_te,y_te),  lr_f.score(X_tr,y_tr_f),  lr_f.score(X_te,y_te_f)],
    [dt0.score(X_tr,y_tr), dt0.score(X_te,y_te), dt0_f.score(X_tr,y_tr_f), dt0_f.score(X_te,y_te_f)],
    [dt2.score(X_tr,y_tr), dt2.score(X_te,y_te), dt2_f.score(X_tr,y_tr_f), dt2_f.score(X_te,y_te_f)],
    [rf.score(X_tr,y_tr),  rf.score(X_te,y_te),  rf_f2.score(X_tr,y_tr_f), rf_f2.score(X_te,y_te_f)],
])

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(data, cmap='RdYlGn', vmin=0.4, vmax=1.0, aspect='auto')
ax.set_xticks([0,1,2,3])
ax.set_xticklabels(['Train Dir', 'Test Dir', 'Train Fill', 'Test Fill'], color='white')
ax.set_yticks(range(4)); ax.set_yticklabels(model_names, color='white')
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{data[i,j]:.2f}', ha='center', va='center', color='white', fontsize=10, fontweight='bold')
ax.set_title('Heatmap de accuracy: verde=bueno, rojo=malo', color='white')
plt.colorbar(im, ax=ax)
fig.patch.set_facecolor('#09090b'); ax.set_facecolor('#18181b')
plt.tight_layout(); plt.show()

---
## Ejercicio 10 — Reflexión final (casa)

*Sin código. Reflexión escrita.*

1. El RandomForest también llega al 100% en training, igual que el DT sin podar. Sin embargo, su test accuracy es 52.7% vs 47.3% del DT. ¿Por qué el RF generaliza mejor si también memoriza?

2. El LR temporal (55.5%) supera al RF (52.7%) con este dataset. ¿En qué condición esperarías que el RF superara al LR?

3. En L8 pasamos de señales micro (dirección del precio) a señales macro (volumen intradia para VWAP). ¿Qué tipo de features esperarías que fueran útiles allí? ¿Se parecen a los que hemos usado aquí?

**Respuesta modelo:**

**1. RF vs DT overfitting:**  
El RF promedia las predicciones de 100 árboles distintos, cada uno entrenado en un bootstrap distinto de los datos y con un subconjunto aleatorio de features por split. Aunque cada árbol individual memoriza (train=100%), los errores de memorización de cada árbol son diferentes e incorrelados. Al promediar, estos errores se cancelan parcialmente, lo que reduce la varianza. Es bagging: alta varianza individual + incorrelación = menor varianza del ensemble.

**2. Cuándo RF > LR:**  
El RF superaría al LR con: (a) muchos más datos de entrenamiento (>5000 muestras), (b) relaciones no lineales o de interacción entre features que LR no puede capturar, (c) features categóricas o con distribuciones no gaussianas. Con 338 muestras y features de distribución razonablemente continua, LR con regularización L2 implícita es más apropiado.

**3. Features para VWAP:**  
Para predecir volumen intradia se usarían: hora del día (efecto intradiario — apertura/cierre tienen más volumen), día de la semana, volumen de los últimos N minutos, indicadores de actividad (spread widening = volatilidad = más volumen). Estos features son temporales pero a una escala diferente (minutos → horas), más relacionados con patrones de comportamiento de mercado que con la microestructura del libro.